# 02 · El Radar: ¿dónde hay presión y dónde hay espacio?
**Torre del Caribe** · Fase 4 (A1 Radar) · Autor: **Brandon Uriel García Sánchez** · UNRC, LCDN 5° semestre 2026-2

| | |
|---|---|
| **Qué hace** | Califica cada lugar, mes por mes, como tranquilo, concurrido o saturado; predice el estado del mes siguiente y el riesgo semanal de que el norte se sature. |
| **Decisiones de Brandon** | Pesos iguales con sensibilidad · percentiles comunes (p50/p90) · escala mín–máx · opción D (ocupación DataTur y componentes de ≥ 2 lugares) · índice comparable · Random Forest · Markov semanal del norte (`docs/decisiones/08-radar.md`). |
| **Código** | `backend/torre/radar/`: `panel.py`, `indice.py`, `prediccion.py`, `markov.py`. |
| **Ecuaciones** | `docs/metodologia/ECUACIONES.md` §2. |
| **Alimenta a** | La campaña: un lugar saturado no se promueve; si se espera que se sature, el anuncio se pausa antes. |

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "backend"))

import pandas as pd
import matplotlib.pyplot as plt
from torre.radar import panel as pn, indice as ix, prediccion as pr, markov as mk
from torre.documento.figuras import estilo_unrc, GUINDA, DORADO, GRIS_TEXTO, GRIS_SUAVE
pd.set_option("display.max_colwidth", 60)
estilo_unrc()

## 1. El panel mensual
Una fila por lugar y mes (enero de 2022 al último mes publicado): los 5 lugares y los destinos del resto del estado,
que dan la escala común. Dos cuidados que se aprendieron al armarlo:
- Los visitantes a zonas arqueológicas se toman del INAH zona por zona: la serie de SITUR-Q es el mismo dato agrupado
  (Chetumal 2025 = Oxtankah + Kohunlich + Dzibanché = 39,459).
- La ocupación de SITUR-Q trae tres filas por mes (disponibles, ocupados y porcentaje): se recalcula como ocupados ÷
  disponibles, nunca se suman ni se promedian porcentajes.

In [2]:
panel = pn.panel_mensual()
print(f"{len(panel):,} filas · {panel.lugar.nunique()} lugares · {panel.periodo.min():%Y-%m} a {panel.periodo.max():%Y-%m}")
pn.cobertura(panel)[["meses_posibles", "llegadas_tren", "cruceristas", "cruces_belice", "visitantes_inah", "ocupacion_pct"]]

825 filas · 15 lugares · 2022-01 a 2026-07


,meses_posibles,llegadas_tren,cruceristas,cruces_belice,visitantes_inah,ocupacion_pct
lugar,,,,,,
Chetumal,55,20,0,54,0,36
Bahía Calderitas–Oxtankah,55,0,0,0,55,0
Ruta arqueológica del sur,55,0,0,0,55,0
Maya Ka'an + Kantemó,55,23,0,0,0,12
Laguna Milagros–Xul-Ha,55,0,0,0,0,0
Bacalar,55,20,0,0,55,12
Cancún,55,31,0,0,55,55
Costa Mujeres,55,0,0,0,55,36
Cozumel,55,0,55,0,55,55


Los 5 lugares no comparten ninguna variable, y la Laguna Milagros no tiene ninguna: el Radar la muestra como
**sin dato oficial**, nunca como tranquila.

## 2. El índice de presión turística
$z=\dfrac{x-\min}{\max-\min}$ con mínimo y máximo de todo el estado; $IPT=$ promedio de las $z$ disponibles (pesos
iguales). Un componente entra solo si lo tienen al menos 2 lugares (sale Belice, que solo tiene Chetumal).

In [3]:
base, cortes, comps = ix.calcular()
print("Componentes del índice:", comps)
print(f"Cortes comunes: p50 = {cortes[0]:.3f}, p90 = {cortes[1]:.3f}")
cancun = base[(base.lugar == "Cancún") & (base.periodo == "2026-07-01")].iloc[0]
print("\nEjemplo a mano, Cancún jul-2026:")
for k in comps:
    if pd.notna(cancun[f"z_{k}"]):
        print(f"  {k:26s} x = {cancun[k]:10.4f}   z = {cancun[f'z_{k}']:.4f}")
print(f"  IPT = promedio = {cancun.ipt:.4f} → {cancun.estado}")

Componentes del índice: ['llegadas_tren_x1000hab', 'cruceristas_x1000hab', 'visitantes_inah_x1000hab', 'llegadas_x_cuarto', 'ocupacion_pct']
Cortes comunes: p50 = 0.258, p90 = 0.766

Ejemplo a mano, Cancún jul-2026:
  llegadas_tren_x1000hab     x =    25.1081   z = 0.0497
  visitantes_inah_x1000hab   x =     0.9608   z = 0.0001
  llegadas_x_cuarto          x =     0.4709   z = 0.0011
  ocupacion_pct              x =    68.7802   z = 0.7214
  IPT = promedio = 0.1931 → tranquilo


In [4]:
# Sensibilidad: ¿cuántos lugar-mes cambian de estado si un peso se mueve ±50 %?
ix.sensibilidad(base, comps)

,componente,factor_peso,cambian_pct,cambian_5_lugares
0,llegadas_tren_x1000hab,0.5,3.1,0
1,llegadas_tren_x1000hab,1.5,1.7,0
2,cruceristas_x1000hab,0.5,2.0,0
3,cruceristas_x1000hab,1.5,0.6,0
4,visitantes_inah_x1000hab,0.5,3.1,1
5,visitantes_inah_x1000hab,1.5,2.3,1
6,llegadas_x_cuarto,0.5,3.1,4
7,llegadas_x_cuarto,1.5,2.0,0
8,ocupacion_pct,0.5,5.7,3
9,ocupacion_pct,1.5,6.0,5


**Prueba de validez.** La primera versión (solo SITUR-Q) calificaba a Cancún "tranquilo" en julio de 2026 porque
SITUR-Q dejó de publicar ocupación en 2025; DataTur marca 68.8 %. Con la opción D (ocupación de DataTur para los 4
lugares que mide y sin componentes de un solo lugar), el norte queda arriba del sur en 2025–2026:

In [5]:
m = base[base.periodo.dt.year >= 2025].groupby("lugar").ipt.mean().round(3)
m.drop(["Costa Mujeres", "Laguna Milagros–Xul-Ha"], errors="ignore").sort_values(ascending=False).to_frame("IPT promedio 2025–2026")

,IPT promedio 2025–2026
lugar,
Mahahual,0.761
Isla Mujeres,0.434
Bacalar,0.319
Tulum,0.259
Cancún,0.206
Cozumel,0.198
Playa del Carmen,0.182
Holbox,0.168
Ruta arqueológica del sur,0.141


## 3. El índice comparable (el que publica el Radar)
En 2025 los lugares que tenían la ocupación de SITUR-Q la pierden y su índice cae (Chetumal 0.54 → 0.05) sin que
llegue menos gente. Por eso cada lugar se mide siempre con las mismas medidas, las que tiene hoy.

In [6]:
t, cortes_c = pr.indice_comparable(base, comps)
print(f"Cortes del índice comparable: p50 = {cortes_c[0]:.3f}, p90 = {cortes_c[1]:.3f}")
t.dropna(subset=["ipt_comparable"]).groupby("lugar").agg(desde=("periodo", "min"), meses=("periodo", "size"), medidas=("medidas", "first"))

Cortes del índice comparable: p50 = 0.186, p90 = 0.756


,desde,meses,medidas
lugar,,,
Bacalar,2024-12-01,20,"llegadas_tren_x1000hab, visitantes_inah_x1000hab, llegad..."
Bahía Calderitas–Oxtankah,2022-01-01,55,visitantes_inah_x1000hab
Cancún,2024-01-01,31,"llegadas_tren_x1000hab, visitantes_inah_x1000hab, llegad..."
Chetumal,2024-12-01,20,"llegadas_tren_x1000hab, llegadas_x_cuarto"
Costa Mujeres,2022-01-01,36,ocupacion_pct
Cozumel,2022-01-01,55,"cruceristas_x1000hab, visitantes_inah_x1000hab, llegadas..."
Holbox,2024-08-01,24,"llegadas_tren_x1000hab, llegadas_x_cuarto"
Isla Mujeres,2022-01-01,55,ocupacion_pct
Mahahual,2022-01-01,55,"cruceristas_x1000hab, llegadas_x_cuarto"


In [7]:
# Gráfica: el índice comparable de los 5 lugares contra la referencia del norte.
fig, ax = plt.subplots(figsize=(9, 4))
lugares = {"Chetumal": GUINDA, "Bahía Calderitas–Oxtankah": "#565393", "Ruta arqueológica del sur": "#58A65D",
           "Maya Ka'an + Kantemó": "#F26E50"}
referencia = {"Cancún": GRIS_TEXTO, "Tulum": GRIS_SUAVE}
for l, c in {**lugares, **referencia}.items():
    s = t[t.lugar == l].set_index("periodo").ipt_comparable.rolling(3, min_periods=1).mean()
    ax.plot(s.index, s.values, color=c, lw=2.4 if l in lugares else 1.6, ls="-" if l in lugares else "--",
            label=l if l in lugares else f"{l} (referencia)")
ax.axhspan(cortes_c[0], cortes_c[1], color=DORADO, alpha=.12, lw=0)
ax.axhspan(cortes_c[1], 1, color=GUINDA, alpha=.08, lw=0)
ax.text(t.periodo.min(), cortes_c[0] + .01, "concurrido", fontsize=8.5, color=GRIS_TEXTO)
ax.text(t.periodo.min(), cortes_c[1] + .01, "saturado", fontsize=8.5, color=GRIS_TEXTO)
ax.set_ylim(0, 1); ax.set_ylabel("Índice de presión (0 a 1)")
ax.set_title("Los cinco lugares se mantienen en la franja tranquila")
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
fig.text(0.01, -0.03, "Promedio móvil de 3 meses. Fuentes: SITUR-Q, INAH, SECTUR-DataTur, Censo 2020. "
         "Cálculo: backend/torre/radar", fontsize=7.5, color=GRIS_TEXTO)
fig.savefig(RAIZ / "docs" / "ejecutivo" / "figuras" / "f08_radar.png")
plt.show()

## 4. ¿Se puede anticipar el mes siguiente?
Tres modelos contra la **persistencia** ("el mes que viene igual que este"), con backtesting de origen móvil: para cada
uno de los últimos 12 meses se reentrena con todo lo anterior y se predice ese mes.

In [8]:
r = pr.correr()
r["metricas"][["modelo", "aciertos", "casos", "cambios_reales", "cambios_acertados", "falsas_alarmas", "f1_macro_origen_movil"]]

,modelo,aciertos,casos,cambios_reales,cambios_acertados,falsas_alarmas,f1_macro_origen_movil
0,Persistencia (línea base),126,156,30,0,0,0.774
1,Regresión logística,130,156,30,8,4,0.793
2,Random Forest,130,156,30,6,2,0.766
3,Gradient Boosting,128,156,30,7,5,0.731


**Lectura honesta:** el estado se repite ocho de cada diez meses, así que la persistencia ya acierta mucho. El
criterio de Brandon elige el modelo que más acierta y, a igualdad, el que más cambios anticipa (tras la auditoría, la
regresión logística: 130 de 156 y 8 de 30 cambios).

**Sesgo** (punto del plan): el mismo backtest separado entre los 5 lugares y el resto del estado.

In [9]:
print("Modelo elegido:", r["mejor"])
r["sesgo"]

Modelo elegido: Regresión logística


,grupo,casos,exactitud_modelo,exactitud_persistencia,cambios_reales,cambios_acertados,falsas_alarmas
0,resto del estado,108,0.778,0.741,28,8,4
1,5 lugares,48,0.958,0.958,2,0,0


Los 5 lugares casi no cambiaron de estado en la prueba: el modelo no se puede validar ahí. Predicción para el mes
siguiente (estimada):

In [10]:
r["prediccion"][["lugar", "estado_hoy", "estado_mes_siguiente_est", "prob_tranquilo", "prob_concurrido", "prob_saturado"]].round(2)

,lugar,estado_hoy,estado_mes_siguiente_est,prob_tranquilo,prob_concurrido,prob_saturado
0,Bacalar,concurrido,concurrido,0.20,0.80,0.0
1,Bahía Calderitas–Oxtankah,tranquilo,tranquilo,1.00,0.00,0.0
2,Cancún,concurrido,tranquilo,0.63,0.37,0.0
3,Chetumal,tranquilo,tranquilo,1.00,0.00,0.0
4,Cozumel,concurrido,tranquilo,0.68,0.32,0.0
5,Holbox,concurrido,tranquilo,0.58,0.42,0.0
6,Isla Mujeres,concurrido,concurrido,0.10,0.90,0.0
7,Mahahual,concurrido,concurrido,0.00,0.60,0.4
8,Maya Ka'an + Kantemó,tranquilo,tranquilo,1.00,0.00,0.0
9,Playa del Carmen,tranquilo,tranquilo,0.77,0.23,0.0


## 5. Cadena de Markov semanal del norte
Estados por percentiles comunes de la ocupación semanal de los 7 centros de DataTur; $\hat p_{ij}=n_{ij}/\sum_j n_{ij}$ y
$\pi_{t+k}=\pi_tP^k$.

In [11]:
mkv = mk.correr()
print(f"Cortes: p50 = {mkv['cortes'][0]:.1f} %, p90 = {mkv['cortes'][1]:.1f} % · transiciones: {int(mkv['n'].to_numpy().sum()):,}")
mkv["P"].round(3)

Cortes: p50 = 71.2 %, p90 = 85.9 % · transiciones: 1,666


hacia,tranquilo,concurrido,saturado
desde,,,
tranquilo,0.922,0.078,0.000
concurrido,0.100,0.831,0.069
saturado,0.012,0.262,0.726


In [12]:
print("Largo plazo (π = πP):", mkv["pi"].round(3).to_dict())
mkv["backtest"]

Largo plazo (π = πP): {'tranquilo': 0.513, 'concurrido': 0.389, 'saturado': 0.098}


,k_semanas,metodo,brier,exactitud,casos
0,1,Markov,0.200,0.887,364
1,1,Persistencia,0.225,0.887,364
2,1,Frecuencia simple,0.544,0.629,364
3,4,Markov,0.389,0.723,364
4,4,Persistencia,0.484,0.758,364
5,4,Frecuencia simple,0.549,0.629,364
6,8,Markov,0.490,0.632,364
7,8,Persistencia,0.676,0.662,364
8,8,Frecuencia simple,0.555,0.629,364


In [13]:
mkv["riesgo"].round(3)

,estado_hoy,ocupacion_hoy_pct,sem_1,sem_2,sem_3,sem_4,sem_5,sem_6,sem_7,sem_8
Akumal,tranquilo,34.4,0.000,0.005,0.013,0.022,0.031,0.040,0.047,0.054
Cancun,tranquilo,67.5,0.000,0.005,0.013,0.022,0.031,0.040,0.047,0.054
Cozumel,tranquilo,56.2,0.000,0.005,0.013,0.022,0.031,0.040,0.047,0.054
Isla Mujeres,tranquilo,48.1,0.000,0.005,0.013,0.022,0.031,0.040,0.047,0.054
Playa Del Carmen,tranquilo,63.9,0.000,0.005,0.013,0.022,0.031,0.040,0.047,0.054
Playacar,concurrido,81.7,0.069,0.107,0.127,0.136,0.139,0.138,0.135,0.132
Riviera Maya,tranquilo,62.1,0.000,0.005,0.013,0.022,0.031,0.040,0.047,0.054


## 6. Minería: el norte comparado con los centros turísticos del país
Clustering jerárquico (Ward) de los centros de DataTur con sus 55 meses completos, por su perfil de ocupación de 12
meses. El número de grupos se elige con la silueta (decisión de Brandon).

In [14]:
from torre.radar import clustering as cl
from scipy.cluster.hierarchy import dendrogram
cls = cl.correr()
print(f"Centros completos: {len(cls['perfiles'])} · excluidos: {len(cls['excluidos'])}")
print("Silueta por k:", {k: round(v, 3) for k, v in cls["siluetas"].items()}, "→ k =", cls["k"])
cls["tabla"]

Centros completos: 55 · excluidos: 41
Silueta por k: {2: 0.45, 3: 0.38, 4: 0.365, 5: 0.3, 6: 0.283, 7: 0.317, 8: 0.306} → k = 2


,centros,nivel_medio,amplitud_media,mes_pico_mas_comun,ejemplos
grupo,,,,,
1,23,64.5,19.9,nov,"Playacar, Akumal, Cancun, Riviera Maya, Los Cabos"
2,32,41.1,17.6,nov,"Morelia, Cozumel, Oaxaca, San Luis Potosi, Guadalajara"


In [15]:
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(cls["Z"], labels=list(cls["perfiles"].index), leaf_font_size=6, color_threshold=cls["Z"][-cls["k"] + 1, 2], ax=ax)
ax.set_title("Centros turísticos del país agrupados por su ocupación mensual (Ward)")
ax.set_ylabel("Distancia de Ward")
plt.show()

## 7. Qué se concluye
1. **Los cinco lugares están tranquilos** en el último mes publicado y el modelo espera que sigan así; la Laguna Milagros
   no tiene dato oficial y así se muestra.
2. **La validez se probó, no se supuso:** la primera versión del índice falló con un caso conocido (Cancún) y se corrigió
   con datos, no a ojo.
3. **El aprendizaje de máquina agrega poco pero algo:** +4 aciertos de 156 sobre la persistencia y 8 cambios anticipados,
   todos en lugares de referencia (los 5 lugares casi no cambiaron de estado).
4. **La cadena de Markov da probabilidades mejor calibradas** que la persistencia a 1, 4 y 8 semanas; su valor es el
   riesgo, no una etiqueta.
5. **El norte de Quintana Roo está en el grupo de los destinos más llenos del país** (k = 2 por silueta).
6. **Limitaciones declaradas:** el sur no tiene ocupación oficial desde 2025 ni datos semanales; la matriz de Markov es
   la misma para los 7 centros; las llegadas por cuarto quedan dominadas por los puertos de crucero; DataTur marca menos
   ocupación que SITUR-Q para el mismo lugar.